# Preliminaries

In [1]:
!pip install numpy  # if numpy is not installed
!pip install scipy  # if scipy is not installed

In [2]:
import numpy as np
from scipy.interpolate import lagrange

# Question 1

In [3]:
# define your samples here
x = np.array([0., 1., 2., 3., 4., 5., 6.])
r = np.array([0., 0., 5., 12., 23., 38., 57.])
#r = np.array([0., 2., 5., 12., 23., 38., 57.]) # use in a second run-through, see comment below
#r = np.array([0., 0., 0., 12., 23., 38., 57.]) # use in a third run-through, see comment below 

num_samples = len(x)

In [4]:
# configure RS-code
(n, k) = (7, 3)
t = (n-k) // 2 # 2t = n-k

We now have $k$ blocks of content and $2t$ parity blocks.
Recall that
* $P(x_i) \cdot E(x_i) = r_i \cdot E(x_i) = Q(x_i)$ for all $i$, and
* $d$ points(samples) will uniquely define a polynomial of degree $d - 1$

In [5]:
# define degrees of polynomials P, E and Q
P_degree = k-1
E_degree = t
Q_degree = (k-1) + t

We also know that $Q(x_i) - r_i \cdot E(x_i) = 0$ for all pairs $(x_i, r_i)$.
This is equal to:
* $q_4 x_i^4 + q_3 x_i^3 + q_2 x_i^2 + q_1 x_i^1 + q_0 - r_i e_1 x_i - r_i e_0 = r_i x_i^2$

Rewriting this as a system of linear equations in matrix form given us:

* $M_1 \cdot C = M_2$

where $C$ is the vector containing all coefficients, and it is the result we want to obtain.

Please define the length of $C$. Recall that the first coefficient of $E(x)$ is a constant 1, thus it is not within the scope of our solution ($C$).

In [6]:
num_coefficients = n  # the length of C

Construct the matrices $M1$ and $M2$ using the sample values $(x_i, r_i)$.

In [7]:
M1 = np.zeros(shape=(num_samples, num_coefficients))
for i in range(num_samples):
    M1[i] = [x[i]**4, x[i]**3, x[i]**2, x[i]**1, x[i]**0, -r[i]*x[i], -r[i]]

In [8]:
M2 = np.zeros(shape=num_samples)
for i in range(num_samples):
    M2[i] = r[i] * x[i]**2

Solve the equation, and obtain all the coefficients ($C$).
Hint: please refer to [numpy.linalg.solve](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html).

In [9]:
C = np.linalg.solve(M1, M2)

In [10]:
# You don't need to round these coefficients. For better display, let's round them here.
C_rounded = np.round(C)
print(C)
print(C_rounded)

[ 2.00000000e+00 -5.00000000e+00  6.00000000e+00 -3.00000000e+00
  0.00000000e+00 -1.00000000e+00  5.03893223e-13]
[ 2. -5.  6. -3.  0. -1.  0.]


In [11]:
Q_coefficients = C[:Q_degree+1]

# At this step, we can concatenate the coefficient 1 at the beginning to obtain the complete list of coefficients for E(x).
E_coefficients = np.hstack((np.ones(1), C[Q_degree+1:]))

print('Q_coefficients:', Q_coefficients)
print('E_coefficients:', E_coefficients)

Q_coefficients: [ 2. -5.  6. -3.  0.]
E_coefficients: [ 1.00000000e+00 -1.00000000e+00  5.03893223e-13]


Finally, we can obtain $P(x)$ by $P(x) = \frac{Q(x)}{E(x)}$. Please refer to [numpy.polydiv](https://numpy.org/doc/stable/reference/generated/numpy.polydiv.html).

In [12]:
P_coefficients = np.polydiv(Q_coefficients, E_coefficients)[0]
print("P_coefficients:", P_coefficients)

P_coefficients: [ 2. -3.  3.]


Now we can reconstruct the true blocks $y$.

In [13]:
y = np.polyval(P_coefficients, x)

# print the result
print("{:<8} {:<8} {:<8}".format("x", "r", "y"))
print("-" * 24)
for i in range(len(x)):
    print("{:<8.2f} {:<8.2f} {:<8.2f}".format(x[i], r[i], y[i]))

x        r        y       
------------------------
0.00     0.00     3.00    
1.00     0.00     2.00    
2.00     5.00     5.00    
3.00     12.00    12.00   
4.00     23.00    23.00   
5.00     38.00    38.00   
6.00     57.00    57.00   


Now rerun the code for the case where one of the blocks is corrupted (see commented out part in the first code cell of Question 1). Compare the result to those you obtained in the first run-through.

Next, rerun the code for the case with three corrupted blocks (second commented out array). Again, compare the results to the first two run-throughs.

# Question 2

In [14]:
x = np.array([ 0., 1., 2., 3., 4., 5.])
y = np.array([10., 5., 8., 2., 7., 4.])
poly = lagrange(x, y)

print(poly)


         5         4         3         2
-0.6333 x + 7.875 x - 34.25 x + 61.12 x - 39.12 x + 10


# Question 3

In [15]:
# define your samples here
x = np.array([0., 1., 2., 3., 4., 5., 6.])
r = np.array([0., 0., 5., 12., 23., 38., 57.])
#r = np.array([3., 2., 5., 12., 23., 38., 57.]) # use in a second run-through, see comment below

num_samples = len(x)

Now we use a technique that inverts the matrix $M_1$ and we solve for the cooefficients by multiplying the inverted matrix to $M_2$: $C = M_1^{-1} M_2$. Re-use the code from above to create the matrices $M_1$ and $M_2$.

In [16]:
(n, k) = (7, 3)
t = (n - k) // 2
Q_degree = (k - 1) + t
num_coefficients = n

M1 = np.zeros(shape=(num_samples, num_coefficients))
for i in range(num_samples):
    M1[i] = [x[i]**4, x[i]**3, x[i]**2, x[i]**1, x[i]**0, -r[i]*x[i], -r[i]]

M2 = np.zeros(shape=num_samples)
for i in range(num_samples):
    M2[i] = r[i] * x[i]**2

First of all, we have to invert $M_1$. Hint: please refer to [numpy.linalg.inv](https://numpy.org/doc/stable/reference/generated/numpy.linalg.inv.html).

In [17]:
M1_inv = np.linalg.inv(M1)

Next, you have to multiply the inverse of $M_1$ to $M_2$. Hint: please refer to [numpy.matmul](https://numpy.org/doc/stable/reference/generated/numpy.matmul.html).

In [18]:
C = np.matmul(M1_inv, M2)
print(C)

[ 2.00000000e+00 -5.00000000e+00  6.00000000e+00 -3.00000000e+00
  0.00000000e+00 -1.00000000e+00  9.09494702e-13]


Compare the coefficients you get from this multiplication with those you get from Question 1.

Now rerun the code by using the array in which none of the blocks are corrupted (commented out section). What kind of result do you get? What kind of result do you get when solving it with linalg.solve? What is happening here? How can you fix this? Hint: please refer to this posting on [stackoverflow](https://stackoverflow.com/questions/28712734/numpy-possible-for-zero-determinant-matrix-to-be-inverted).

Coefficients are the same.

When there are no errors, the true error locator polynomial is just E(x) = 1 (degree 0, no roots). But our system was built assuming t=2 errors, so it forces E to have degree 2. This means there are infinitely many valid solutions, the system is underdetermined and M1 becomes singular (determinant = 0), so it cannot be inverted.

We can use pinv (pseudoinverse) instead of inv bc it gives the best approximate solution instead of crashing

In [23]:
print("\n\nSecond run-through with different r values:\n")
r_second = np.array([0., 2., 5., 12., 23., 38., 57.]) # use in a second run-through

(n, k) = (7, 3)
t = (n-k) // 2 # 2t = n-k

# define degrees of polynomials P, E and Q
P_degree_second = k-1
E_degree_second = t
Q_degree_second = (k-1) + t

num_coefficients_second = n  # the length of C

M1_second_run = np.zeros(shape=(num_samples, num_coefficients_second))
for i in range(num_samples):
    M1_second_run[i] = [x[i]**4, x[i]**3, x[i]**2, x[i]**1, x[i]**0, -r_second[i]*x[i], -r_second[i]]

M2_second_run = np.zeros(shape=num_samples)
for i in range(num_samples):
    M2_second_run[i] = r_second[i] * x[i]**2
    
C_sec = np.linalg.solve(M1_second_run, M2_second_run)

C_rounded_sec = np.round(C_sec)
print(C_sec)
print(C_rounded_sec)

Q_coefficients_second = C[:Q_degree_second+1]

# At this step, we can concatenate the coefficient 1 at the beginning to obtain the complete list of coefficients for E(x).
E_coefficients_second = np.hstack((np.ones(1), C[Q_degree_second+1:]))

print('Q_coefficients:', Q_coefficients_second)
print('E_coefficients:', E_coefficients_second)

P_coefficients_second = np.polydiv(Q_coefficients_second, E_coefficients_second)[0]
print("P_coefficients:", P_coefficients_second)

y = np.polyval(P_coefficients_second, x)

# print the result
print("{:<8} {:<8} {:<8}".format("x", "r", "y"))
print("-" * 24)
for i in range(len(x)):
    print("{:<8.2f} {:<8.2f} {:<8.2f}".format(x[i], r[i], y[i]))

x_second = np.array([ 0., 1., 2., 3., 4., 5.])
y_second = np.array([10., 5., 8., 2., 7., 4.])
poly_second = lagrange(x_second, y_second)

print(poly_second)

# define your samples here
x = np.array([0., 1., 2., 3., 4., 5., 6.])
#r = np.array([0., 0., 5., 12., 23., 38., 57.])
r = np.array([3., 2., 5., 12., 23., 38., 57.]) # use in a second run-through, see comment below

num_samples = len(x)





Second run-through with different r values:

[ 2.00000000e+00 -9.01722788e+00  1.20258418e+01 -9.02584182e+00
  0.00000000e+00 -3.00861394e+00  6.97847186e-14]
[ 2. -9. 12. -9.  0. -3.  0.]
Q_coefficients: [ 2. -5.  6. -3.  0.]
E_coefficients: [ 1.00000000e+00 -1.00000000e+00  9.09494702e-13]
P_coefficients: [ 2. -3.  3.]
x        r        y       
------------------------
0.00     3.00     3.00    
1.00     2.00     2.00    
2.00     5.00     5.00    
3.00     12.00    12.00   
4.00     23.00    23.00   
5.00     38.00    38.00   
6.00     57.00    57.00   
         5         4         3         2
-0.6333 x + 7.875 x - 34.25 x + 61.12 x - 39.12 x + 10


In [21]:
print("\nThird run-through with different r values:\n")
r = np.array([0., 0., 0., 12., 23., 38., 57.]) # use in a third run-through, see comment below 

(n, k) = (7, 3)
t = (n-k) // 2 # 2t = n-k

# define degrees of polynomials P, E and Q
P_degree_third = k-1
E_degree_third = t
Q_degree_third = (k-1) + t

num_coefficients_third = n  # the length of C

M1_third_run = np.zeros(shape=(num_samples, num_coefficients_third))
for i in range(num_samples):
    M1_third_run[i] = [x[i]**4, x[i]**3, x[i]**2, x[i]**1, x[i]**0, -r[i]*x[i], -r[i]]

M2_third_run = np.zeros(shape=num_samples)
for i in range(num_samples):
    M2_third_run[i] = r[i] * x[i]**2

C_third = np.linalg.solve(M1_third_run, M2_third_run)

C_rounded_third = np.round(C_third)
print(C_third)
print(C_rounded_third)

Q_coefficients_third = C[:Q_degree_third+1]

# At this step, we can concatenate the coefficient 1 at the beginning to obtain the complete list of coefficients for E(x).
E_coefficients_third = np.hstack((np.ones(1), C[Q_degree_third+1:]))

print('Q_coefficients:', Q_coefficients_third)
print('E_coefficients:', E_coefficients_third)

P_coefficients_third = np.polydiv(Q_coefficients_third, E_coefficients_third)[0]
print("P_coefficients:", P_coefficients_third)

y = np.polyval(P_coefficients_third, x)

# print the result
print("{:<8} {:<8} {:<8}".format("x", "r", "y"))
print("-" * 24)
for i in range(len(x)):
    print("{:<8.2f} {:<8.2f} {:<8.2f}".format(x[i], r[i], y[i]))

x_third = np.array([ 0., 1., 2., 3., 4., 5.])
y_third = np.array([10., 5., 8., 2., 7., 4.])
poly_third = lagrange(x_third, y_third)

print(poly_third)

# define your samples here
x = np.array([0., 1., 2., 3., 4., 5., 6.])
#r = np.array([0., 0., 5., 12., 23., 38., 57.])
r = np.array([3., 2., 5., 12., 23., 38., 57.]) # use in a second run-through, see comment below

num_samples = len(x)


Third run-through with different r values:

[  1.58333333 -47.5        131.41666667 -85.5          0.
 -26.          50.        ]
[  2. -48. 131. -86.   0. -26.  50.]
Q_coefficients: [ 2. -5.  6. -3.  0.]
E_coefficients: [ 1.00000000e+00 -1.00000000e+00  9.09494702e-13]
P_coefficients: [ 2. -3.  3.]
x        r        y       
------------------------
0.00     0.00     3.00    
1.00     0.00     2.00    
2.00     0.00     5.00    
3.00     12.00    12.00   
4.00     23.00    23.00   
5.00     38.00    38.00   
6.00     57.00    57.00   
         5         4         3         2
-0.6333 x + 7.875 x - 34.25 x + 61.12 x - 39.12 x + 10
